# SAR Automatic Target Recognition — Must-Run Experiments

**Environment**: Vast.ai GPU instance  
**Dataset**: MSTAR (10 vehicle classes, ~1000 SAR images)  
**Model**: CNN + GNN few-shot open-set recognition  

**Experiments in this notebook** (Tier 1 — minimum viable paper):

| # | Experiment | Purpose |
|---|-----------|---------|
| Exp 1 | FSL+GNN 3-way 20-shot, λ=0 | Reproduce Zhou et al. baseline |
| Exp 2 | CNN closed-set baseline | Prove FSL is worth it |
| Exp 3 | FSL+GNN + physics loss λ=0.1 | **Main contribution** |
| Exp 4 | Physics λ ablation (0.01, 0.5) | Sensitivity analysis |

Run cells top-to-bottom. Each section is self-contained and can be re-run independently.

---

## 0. GPU Check

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — check instance type')

Sun Apr 26 23:47:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.21                 Driver Version: 596.21         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   66C    P0             62W /  137W |    1540MiB /   8188MiB |     68%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

PyTorch version : 2.1.2
CUDA available  : True
GPU             : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM            : 8.6 GB


## 1. Environment Setup

In [3]:
import sys

# Show exactly which Python the kernel is using
print('Kernel Python   :', sys.executable)
print('Python version  :', sys.version)
print()
print('This is the Python that pip must install into.')
print('The next cell uses sys.executable to guarantee that.')

Kernel Python   : C:\Users\husse\.conda\envs\sar_playground\python.exe
Python version  : 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]

This is the Python that pip must install into.
The next cell uses sys.executable to guarantee that.


## 3. MSTAR Dataset

In [4]:
import os

DATASET_DIR = os.path.join(os.getcwd(), 'MSTAR')

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f'Dataset not found at {DATASET_DIR} — run the download cell first')

classes = sorted(os.listdir(DATASET_DIR))
print(f'Found {len(classes)} classes:\n')
total = 0
for cls in classes:
    cls_path = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    n = len(os.listdir(cls_path))
    total += n
    marker = '  [UNSEEN - held out]' if cls == 'T72' else ''
    print(f'  {cls:<15} {n:>4} images{marker}')

print(f'\nTotal images : {total}')
assert 'T72' in classes, 'T72 class not found — check dataset structure'
print('\nDataset structure OK')

Found 10 classes:

  2S1              573 images
  BMP2             428 images
  BRDM_2           572 images
  BTR60            451 images
  BTR70            429 images
  D7               573 images
  T62              571 images
  T72              428 images  [UNSEEN - held out]
  ZIL131           573 images
  ZSU_23_4         573 images

Total images : 5171

Dataset structure OK


In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# import os, random

# DATASET_DIR = os.path.join(os.getcwd(), 'MSTAR')
# classes = sorted([c for c in os.listdir(DATASET_DIR)
#                   if os.path.isdir(os.path.join(DATASET_DIR, c))])

# fig, axes = plt.subplots(2, 5, figsize=(14, 6))
# fig.suptitle('MSTAR Dataset — One sample per class', fontsize=13)

# for ax, cls in zip(axes.flat, classes):
#     cls_dir = os.path.join(DATASET_DIR, cls)
#     img_file = random.choice(os.listdir(cls_dir))
#     img = mpimg.imread(os.path.join(cls_dir, img_file))
#     ax.imshow(img, cmap='gray')
#     ax.set_title(cls + (' [UNSEEN]' if cls == 'T72' else ''), fontsize=9)
#     ax.axis('off')

# plt.tight_layout()
# os.makedirs('results', exist_ok=True)
# plt.savefig('results/dataset_preview.png', dpi=120, bbox_inches='tight')
# plt.show()

## 4. Create Output Directories

In [5]:
import os
for d in ['model', 'log', 'results', 'gan_output', 'model/gan']:
    os.makedirs(d, exist_ok=True)
print('Output directories ready')

Output directories ready


## 5. Shape Smoke Test

Verifies CNN and GNN produce the correct tensor shapes before committing to a long training run.

In [6]:
import sys, os
PROJECT_DIR = os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
from cnn import EmbeddingCNN
from gnn import GNN_module

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running smoke test on: {device}\n')

# CNN: (batch=4, 1, 100, 100) -> (batch, 64)
cnn = EmbeddingCNN(image_size=100, cnn_feature_size=64,
                   cnn_hidden_dim=32, cnn_num_layers=4).to(device)
x   = torch.rand(4, 1, 100, 100, device=device)
out = cnn(x)
assert out.shape == (4, 64), f'CNN shape error: {out.shape}'
print(f'CNN output shape  : {list(out.shape)}  OK')

# GNN: 3-way 20-shot, batch=8
nway, shots, batch = 3, 20, 8
D      = 64 + nway + 1
N      = nway * shots + 1
nodes  = torch.rand(batch, N, D, device=device)
gnn    = GNN_module(nway=nway, input_dim=D, hidden_dim=16).to(device)
logits = gnn(nodes)
assert logits.shape == (batch, nway + 1), f'GNN shape error: {logits.shape}'
print(f'GNN output shape  : {list(logits.shape)}  OK  (batch x C+1 logits)')

# DataLoader
from data import self_DataLoader
dl = self_DataLoader(root='MSTAR', nway=3, unseen_class='T72', unseen_ratio=1.0)
batch_data = dl.load_tr_batch(batch_size=4, nway=3, num_shots=5)
print(f'DataLoader batch  : {[list(b.shape) for b in batch_data[:4]]}  OK')

print('\nAll smoke tests PASSED — safe to train')

Running smoke test on: cuda

CNN output shape  : [4, 64]  OK
GNN output shape  : [8, 4]  OK  (batch x C+1 logits)


ImportError: cannot import name 'ExportOptions' from 'torch.onnx._internal.exporter' (C:\Users\husse\.conda\envs\sar_playground\lib\site-packages\torch\onnx\_internal\exporter\__init__.py)

## 7. Experiment 2 — CNN Baseline (Full MSTAR Dataset)

Standard CNN classifier trained on all MSTAR images (10-way) — establishes the closed-set baseline.  
Needed to show FSL+GNN is worth using over a plain CNN.

**Expected runtime**: ~10-15 min each (RTX 5080, 16GB VRAM)

In [7]:
# Full-data CNN baseline (ALL images from all MSTAR classes)
import sys
!{sys.executable} main.py \
    --data_root     MSTAR \
    --model_type    cnn \
    --nway          10 \
    --shots         100 \
    --batch_size    64 \
    --lr            0.001 \
    --max_iteration 100000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    15 \
    --affix         full_all_data \
    --eval_output   results \
    --save \
    --use_gpu       0

GPU: NVIDIA GeForce RTX 4060 Laptop GPU, VRAM: 8.6GB
Optimizing for low-VRAM GPU (< 12GB)
  - batch_size=64, eval_batch_size=2
  - eval_sample=2000, gradient_checkpointing=True
unseen class: T72 (idx 7)
full_train_num: 3790
full_test_num:  953
few_data_num:   428


todo             : train
dataset          : MSTAR
model_type       : cnn
use_gpu          : 0
seed             : 1
batch_size       : 64
lr               : 0.001
max_iteration    : 100000
log_interval     : 100
eval_interval    : 500
early_stop       : 15
early_stop_pretrain : 10
test_dir         : 
data_root        : MSTAR
log_root         : log
model_root       : model
affix            : full_all_data
save             : True
load             : False
load_dir         : model/3way_20shot_gnn_
output_dir       : output
output_name      : output.txt
nway             : 10
shots            : 100
freeze_cnn       : False
unseen_class     : T72
unseen_ratio     : 1.0
warmup_iters     : 2000
gan_augment      : False
gan_output_dir   : gan_output
physics_lambda   : 0.0
augment_rotation : False
augment_speckle  : False
speckle_sigma    : 0.1
eval_only        : False
eval_output      : results
baseline_kshot   : False
amp              : True
eval_batch_size  : 2
gradient_checkpointing : True
eva

In [ ]:
# CNN baseline with entire dataset — no few-shot limitation
import sys
!{sys.executable} main.py \
    --data_root     MSTAR \
    --model_type    cnn \
    --nway          10 \
    --shots         150 \
    --batch_size    64 \
    --lr            0.001 \
    --max_iteration 100000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    15 \
    --affix         full_all_data_large \
    --eval_output   results \
    --save \
    --use_gpu       0

## 9. Experiment 4 — Physics-Informed Regularization (Full MSTAR Dataset, λ=0.1)

Adds the intra-class embedding variance penalty to the training loss (λ=0.1), trained on all MSTAR images.  
Same GNN architecture as Exp 3 — the only change is the physics loss term.

**Physics loss**: penalizes high variance among same-class support embeddings, encoding the physical expectation  
that the same SAR vehicle produces consistent features despite speckle noise variation.

**Target**: higher overall/seen/unseen accuracy than Exp 3 baseline  
**Expected runtime**: ~40-50 min (RTX 5080, 16GB VRAM)

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root      MSTAR \
    --model_type     gnn \
    --nway           9 \
    --shots          100 \
    --unseen_class   T72 \
    --unseen_ratio   1.0 \
    --warmup_iters   2000 \
    --physics_lambda 0.1 \
    --batch_size     32 \
    --lr             0.001 \
    --max_iteration  150000 \
    --log_interval   100 \
    --eval_interval  500 \
    --early_stop     15 \
    --affix          9way_full_physics_0.1 \
    --eval_output    results \
    --save \
    --use_gpu        0

In [ ]:
import sys
!{sys.executable} main.py \
    --data_root      MSTAR \
    --model_type     gnn \
    --nway           9 \
    --shots          100 \
    --unseen_class   T72 \
    --unseen_ratio   1.0 \
    --warmup_iters   2000 \
    --physics_lambda 0.0 \
    --batch_size     32 \
    --lr             0.001 \
    --max_iteration  150000 \
    --log_interval   100 \
    --eval_interval  500 \
    --early_stop     15 \
    --affix          9way_full_baseline \
    --eval_output    results \
    --save \
    --use_gpu        0

## 8. Experiment 3 — GNN Baseline (Full MSTAR Dataset, λ=0)

FSL+GNN trained on all MSTAR images (9-way train + 1-way unseen T72) with no physics regularization (λ=0).  
This is the baseline to compare physics-loss ablation against.

**Expected runtime**: ~30-40 min (RTX 5080, 16GB VRAM)

## 10. Experiment 5 — Physics Lambda Ablation (Full MSTAR Dataset)

Sweeps λ ∈ {0.01, 0.5} on all MSTAR data — the λ=0 (Exp 3) and λ=0.1 (Exp 4) results are already available.  
Compare all four to find the optimal regularization strength.

**Expected runtime**: ~40 min each (~80 min total, RTX 5080)

In [ ]:
import sys, subprocess

for lam in ['0.01', '0.5']:   # λ=0 needs baseline; λ=0.1 is main result above
    print(f'\n{"="*60}')
    print(f'  Physics lambda = {lam} (full data, 9-way)')
    print(f'{"="*60}\n', flush=True)

    proc = subprocess.Popen(
        [sys.executable, 'main.py',
         '--data_root',      'MSTAR',
         '--model_type',     'gnn',
         '--nway',           '9',
         '--shots',          '100',
         '--unseen_class',   'T72',
         '--unseen_ratio',   '1.0',
         '--warmup_iters',   '2000',
         '--physics_lambda', lam,
         '--batch_size',     '32',
         '--lr',             '0.001',
         '--max_iteration',  '150000',
         '--log_interval',   '100',
         '--eval_interval',  '500',
         '--early_stop',     '15',
         '--affix',          f'9way_full_physics_{lam}',
         '--eval_output',    'results',
         '--save',
         '--use_gpu',        '0'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

print('\nAblation complete — check results/ for all lambda configs')

## 11. Evaluate a Saved Checkpoint (No Re-Training)

Load any saved model from full-data training and run evaluation without re-training.  
Useful for re-evaluating after the instance is restarted.

In [ ]:
import sys
# Change --load_dir to point at whichever full-data checkpoint you want to evaluate
!{sys.executable} main.py \
    --data_root    MSTAR \
    --model_type   gnn \
    --nway         9 \
    --shots        100 \
    --unseen_class T72 \
    --load \
    --load_dir     model/9way_full_baseline/ \
    --eval_only \
    --eval_output  results \
    --use_gpu      0

## 12. Results: Comparison Table and Charts (Full MSTAR Data)

In [ ]:
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import json, glob
from evaluate import print_comparison_table

report_files = sorted(glob.glob('results/*.json'))

if not report_files:
    print('No result JSON files found yet — run training experiments first')
else:
    print(f'Found {len(report_files)} report(s):\n')
    for f in report_files:
        print(' ', f)
    print()
    table = print_comparison_table(report_files)
    with open('results/comparison_table.md', 'w') as f:
        f.write('# Results Comparison\n\n' + table)
    print('\nSaved to results/comparison_table.md')

In [ ]:
# Bar chart: overall / seen / unseen accuracy per experiment
import json, glob, os
import matplotlib.pyplot as plt

report_files = sorted(glob.glob('results/*.json'))

if report_files:
    configs, overall, seen, unseen = [], [], [], []
    for path in report_files:
        with open(path) as f:
            d = json.load(f)
        configs.append(d.get('config', os.path.basename(path).replace('.json', '')))
        overall.append(d.get('overall_accuracy', 0))
        seen.append(d.get('seen_accuracy', 0))
        unseen.append(d.get('unseen_accuracy', 0))

    x, w = range(len(configs)), 0.25
    fig, ax = plt.subplots(figsize=(max(8, len(configs) * 2), 5))
    ax.bar([i - w for i in x], overall, w, label='Overall', color='steelblue')
    ax.bar([i     for i in x], seen,    w, label='Seen',    color='seagreen')
    ax.bar([i + w for i in x], unseen,  w, label='Unseen',  color='tomato')
    ax.axhline(95, linestyle='--', color='seagreen', alpha=0.4, label='Seen target 95%')
    ax.axhline(62, linestyle='--', color='tomato',   alpha=0.4, label='Unseen target 62%')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Accuracy by Experiment')
    ax.set_xticks(list(x))
    ax.set_xticklabels(configs, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 110)
    ax.legend()
    plt.tight_layout()
    plt.savefig('results/accuracy_chart.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No JSON reports found')

In [ ]:
# Confusion matrix for the 3-way 20-shot baseline
import json, glob
import numpy as np
import matplotlib.pyplot as plt

reports = glob.glob('results/3way_20shot_gnn_*.json')
if reports:
    with open(sorted(reports)[-1]) as f:
        d = json.load(f)

    cm = np.array(d['confusion_matrix'])
    names = list(d['per_class'].keys())
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — 3-way 20-shot GNN')
    for i in range(len(names)):
        for j in range(len(names)):
            v = cm_norm[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if v > 0.5 else 'black')
    plt.tight_layout()
    plt.savefig('results/confusion_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No baseline report found — run Experiment 1 first')

## 12. Save Results to Google Drive

Run **one** of the three options below before your Vast.ai instance is terminated.

In [ ]:
# ── OPTION A: Vast.ai — zip everything and download via Jupyter file browser ─
import shutil, os, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_path  = f'/workspace/sar_results_{timestamp}'
shutil.make_archive(zip_path, 'zip', os.getcwd())

final = zip_path + '.zip'
print(f'Archive: {final}  ({os.path.getsize(final)/1e6:.1f} MB)')
print('Download it from the Jupyter file browser (left panel)')

In [ ]:
# ── OPTION B: Vast.ai — sync to Google Drive via rclone ────────────────────
# First-time setup (run once in the Vast.ai terminal, NOT in this cell):
#   rclone config
#   --> choose Google Drive, name the remote 'gdrive'

import sys
GDRIVE_DEST = 'gdrive:SAR_results'  # <-- change folder name if needed

!rclone copy results/ {GDRIVE_DEST}/results/ --progress
!rclone copy model/   {GDRIVE_DEST}/model/   --progress
!rclone copy log/     {GDRIVE_DEST}/log/     --progress
print('Upload complete')

In [ ]:
# ── OPTION C: Google Colab only — mount Drive and copy ─────────────────────
from google.colab import drive
import shutil, os

drive.mount('/gdrive')
DRIVE_DEST = '/gdrive/MyDrive/SAR_results'  # <-- change if needed
os.makedirs(DRIVE_DEST, exist_ok=True)

for folder in ['results', 'log', 'model']:
    src = os.path.join(os.getcwd(), folder)
    if os.path.exists(src):
        shutil.copytree(src, os.path.join(DRIVE_DEST, folder), dirs_exist_ok=True)
        print(f'Copied {folder}/')
print('Done')

---
## Quick Reference — Full MSTAR Dataset Experiments

| Section | Experiment | Classes | Shots | Est. GPU time (RTX 5080) |
|---------|-----------|---------|-------|--------------------------|
| §5  | Smoke test | — | — | < 1 min |
| §7  | **Exp 2a**: CNN baseline (10-way, all data) | 10 | 100 | ~10 min |
| §7  | **Exp 2b**: CNN baseline (10-way, extended) | 10 | 150 | ~15 min |
| §8  | **Exp 3**: GNN baseline (9-train+T72, λ=0) | 9+1 | 100 | ~40 min |
| §9  | **Exp 4**: GNN + physics (9-train+T72, λ=0.1) — **main result** | 9+1 | 100 | ~50 min |
| §10 | **Exp 5**: Physics λ ablation (0.01, 0.5) | 9+1 | 100 | ~80 min total |
| §11 | Eval-only on saved checkpoint | 9+1 | 100 | ~5 min |
| §12 | Comparison table + charts | — | — | < 1 min |

**Total**: ~3 hours for all full-data experiments.

### Updated results table to fill in
| Model | λ | Overall | Seen | Unseen |
|-------|----|---------|------|--------|
| CNN baseline (10-way, 100-shot) | — | | | |
| CNN baseline (10-way, 150-shot) | — | | | |
| GNN (9-train+T72, baseline) | 0 | | | |
| GNN + physics | 0.01 | | | |
| GNN + physics | **0.1** | | | |
| GNN + physics | 0.5 | | | |

### Key output paths
```
model/full_all_data/                    Exp 2a best checkpoint (CNN 10-way)
model/full_all_data_large/              Exp 2b best checkpoint (CNN 10-way extended)
model/9way_full_baseline/               Exp 3 best checkpoint (GNN baseline)
model/9way_full_physics_0.1/            Exp 4 best checkpoint (main result)
model/9way_full_physics_0.01/           Exp 5 ablation checkpoint
model/9way_full_physics_0.5/            Exp 5 ablation checkpoint
results/*.json                          per-experiment metrics (full-data)
results/comparison_table.md             printable results table
results/accuracy_chart.png              bar chart
log/*/                                  full training logs
```

### Tips for full-data training
- Monitor GPU memory with `nvidia-smi` — batch_size=32 uses ~12-14 GB on RTX 5080
- Increase `--early_stop` patience (15) to allow more exploration with large datasets
- Higher `--max_iteration` (150000) needed to converge on full data vs. few-shot episodes
- T72 is held-out unseen class for evaluating open-set recognition